in feature

In [0]:
from pyspark.sql.functions import current_timestamp,lit

df_silver = spark.readStream.table("workspace.streaming.ecommerce_silver")

In [0]:
def scd2_merge(batch_df, batch_id):

    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, col, desc
    from delta.tables import DeltaTable

    target_table = "workspace.streaming.ecommerce_gold_scd2"

    # ✅ Deduplicate using event time
    window = Window.partitionBy("order_id").orderBy(desc("kafka_timestamp"))

    batch_df = (
        batch_df
        .withColumn("rank", row_number().over(window))
        .filter("rank = 1")
        .drop("rank")
    )

    batch_df = batch_df \
        .withColumn("start_date", current_timestamp()) \
        .withColumn("end_date", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True))

    if not spark.catalog.tableExists(target_table):
        batch_df.limit(0).write.format("delta").saveAsTable(target_table)

    target = DeltaTable.forName(spark, target_table)
    existing = spark.table(target_table)
    existing_current = existing.filter("is_current = true")  # ✅ CRITICAL FIX

    # ✅ Step 1: Changed
    changed_df = batch_df.alias("source").join(
        existing_current.alias("target"),
        "order_id"
    ).where("source.amount <> target.amount")

    # ✅ Step 2: Expire
    target.alias("target").merge(
        changed_df.alias("source"),
        "target.order_id = source.order_id AND target.is_current = true"
    ).whenMatchedUpdate(
        set={
            "end_date": "current_timestamp()",
            "is_current": "false"
        }
    ).execute()

    # ✅ Step 3: Insert
    new_records_df = batch_df.alias("source").join(
        existing_current.alias("target"),
        "order_id",
        "left"
    ).where(
        "target.order_id IS NULL OR source.amount <> target.amount"
    ).select("source.*")

    new_records_df.write.format("delta").mode("append").saveAsTable(target_table)

In [0]:
query = df_silver.writeStream \
    .foreachBatch(scd2_merge) \
    .outputMode("update") \
    .option("checkpointLocation", "/Volumes/workspace/streaming/checkpoints/ecommerce_pipeline/gold") \
    .trigger(availableNow=True) \
    .start()

In [0]:
%sql
select count(*) from workspace.streaming.ecommerce_gold_scd2;

In [0]:
%sql
SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id, kafka_timestamp
            ORDER BY kafka_timestamp DESC
        ) AS rn
FROM workspace.streaming.ecommerce_gold_scd2

In [0]:
%sql
select * from workspace.streaming.ecommerce_gold_scd2;